# 06 开放词检测入门

## 学习目标

- 使用 `eco_detect_objects` 检测物品。
- 可视化检测框。
- 理解 OVD 与 SHOE endpoint。
- 根据检测数量和 bbox 判断是否适合进入抓取流程。

参考文献：文档第 6.1.1 章。

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import DEFAULT_WS_URL

WS_URL = DEFAULT_WS_URL
print("WS_URL =", WS_URL)

In [ ]:
CONNECT_ROBOT = False
CAPTURE_IMAGE = False
DETECT_OBJECTS = False
LABELS = ["鞋子"]
USE_SHOE_ENDPOINT = True

In [ ]:
from helpers import connect_robot, disconnect_safely, show_rgb, draw_bboxes, summarize_detections
from bajie_sdk import CameraType, DetectObjectsRequest, OvdEndpoint

robot = connect_robot(WS_URL) if CONNECT_ROBOT else None
view = None
if robot and CAPTURE_IMAGE:
    view = robot.eco_captureImages(CameraType.HEAD, timeout_sec=30.0)
    show_rgb(view, "Image for detection")
else:
    print("Set CONNECT_ROBOT and CAPTURE_IMAGE to True first.")

In [ ]:
detections = []
if robot and view is not None and DETECT_OBJECTS:
    entry = OvdEndpoint.SHOE if USE_SHOE_ENDPOINT else OvdEndpoint.OVD
    resp = robot.eco_detect_objects(
        DetectObjectsRequest(
            rgb_image=view.rgb_image.img,
            labels=LABELS,
            entry=entry,
        ),
        timeout=15.0,
    )
    detections = list(resp.items)
    summarize_detections(detections)
    draw_bboxes(view, detections, "Detection results")
else:
    print("Set DETECT_OBJECTS = True after capturing an image.")

In [ ]:
disconnect_safely(robot)

## 机器人习惯

- 标签词尽量具体，比如“鞋子”“杯子”“玩具汽车”通常比“东西”更好。
- 检测依赖图片质量；先让目标出现在画面中，再检测。
- SHOE endpoint 是鞋子专用检测，其他开放词通常走 OVD。

## 故障排查卡片

- 没检测到：换标签、换角度、降低阈值、确认目标在画面里。
- 父类标签可能会自动展开为子类，但具体任务中仍建议用明确标签。
- 服务未配置或不可达时，感知接口可能超时或报错。

## 小练习

换一个标签，观察检测框数量和位置是否变化。